# AI2-THOR rollout evaluation — error analysis

Aggregates `eval_rollout_thor.py` record batteries (per-(arm, clip) `.npz` of
per-step scalars). All comparisons are PAIRED over the clips common to every
arm, so a partial battery is fine — coverage is printed at load. Any
`CLIP_IDX` printed below replays in `rollout_ai2thor.ipynb` (same ordering).
Machinery: `far.analysis.thor_eval` (one function per cell).

Needs: a record battery under `results/rollout_eval/<battery>/<arm>/*.npz`,
written by `scripts/world/eval_rollout_thor.py` (one sub-directory per arm;
`EVAL_DIR` below names the battery). The stale-reference cells additionally
decode the episodes' latents with the SDXL VAE (`models/sdxl-vae-fp16-fix/`,
or its Hugging Face copy) on the default CUDA device — pass `device=` to
`E.build_stale_ref` to choose another one; nothing else needs a GPU.

In [ ]:
# %%
import os
from far.constants import ROOT
os.chdir(ROOT)
os.environ.setdefault("FAR_RESULT_ROOT", os.path.join(ROOT, "results"))
from far.analysis import thor_eval as E

# the battery directory (eval_rollout_thor.py's output_dir): one sub-dir of .npz records per arm
EVAL_DIR = "results/rollout_eval/thor3_0700000_cs12_stride4"

recs = E.load_records(EVAL_DIR)

In [ ]:
# %%
# Whole-battery summary: frame-wise metrics (all steps), in-mask metrics
# (mask-valid steps), and the copy rate (prediction closer to the top-1
# retrieved frame than to GT inside the change region).
E.summary_table(recs)

In [ ]:
# %%
# Frame-wise error vs normalized retrace progress (mean ± 95% CI over clips).
METRIC = "lpips"   # psnr | lpips | dreamsim
E.plot_frac_curves(recs, METRIC)

In [ ]:
# %%
# Event-aligned: error vs signed seconds to the nearest reveal (0 = the open).
# The right panel restricts to the change region — the state-rendering cost
# that whole-frame metrics smear out.
E.plot_event_aligned(recs, metrics=("lpips", "in_mse"))

In [ ]:
# %%
# Copy signature per mask-valid step: below the diagonal = the arm reproduced
# what its memory handed it instead of the true change.
E.plot_copy_signature(recs)

In [ ]:
# %%
# Binary STATE-ERROR proxy: a mask-valid step fails when its change-region
# crop-LPIPS exceeds TAU. Sweep -> threshold choice; tables at TAU: overall,
# CONTAINER (within +-REVEAL_WINDOW s of an eval reveal) vs SURFACE strata,
# and the container stratum by station. SWING_MARGIN excludes the
# door-ANIMATION steps just before the settled open (a wrong swing graphic
# inflates the crop without saying anything about the rendered state; the
# animation spike stays visible in the event-aligned cell above).
TAU = 0.15
REVEAL_WINDOW = 3.0   # seconds around an eval-leg open counted as "container"
SWING_MARGIN  = 0.8   # seconds of pre-open animation excluded (0 = count all)
E.state_error_rate(recs, tau=TAU, reveal_window=REVEAL_WINDOW, swing_margin=SWING_MARGIN)

In [ ]:
# %%
# State error as a bar chart: the container stratum of the proxy above split
# by station type, plus the SURFACE stratum (every other masked change); bars
# in the order temporal / worldmem / ours — clip-mean failure rate with a 95%
# CI over clips. Same TAU / REVEAL_WINDOW / SWING_MARGIN as the table.
_ = E.plot_state_error_by_station(recs, tau=TAU, reveal_window=REVEAL_WINDOW, swing_margin=SWING_MARGIN)


In [ ]:
# %%
# STALE-REFERENCE diagnostic: d(generated, GT) < d(GT, stale) on the change-region
# crop, `stale` = the exact-view exploration counterpart the .change mask was
# diffed against (the old state). d(GT, stale) is arm-independent, so it is built
# ONCE per (clip, step) into <EVAL_DIR>/stale_ref/ (~20 min on a GPU; later calls
# just load it). MIN_REF drops steps whose stale frame is nearly identical to GT in
# the crop (a change too small to judge — the test is decoder noise there); the
# binned rows show success as a function of how visible the change is.
MIN_REF = 0.05
REF = E.build_stale_ref(EVAL_DIR)
E.stale_margin_table(recs, REF, reveal_window=REVEAL_WINDOW, swing_margin=SWING_MARGIN, min_ref=MIN_REF)


In [ ]:
# %%
# The same bar chart with the per-step stale threshold instead of TAU (steps with
# d(GT, stale) < MIN_REF excluded).
_ = E.plot_state_error_by_station(recs, reveal_window=REVEAL_WINDOW, swing_margin=SWING_MARGIN, ref=REF, min_ref=MIN_REF)


In [ ]:
# %%
# Stale-margin error vs the SIZE of the change, surface (left) vs container (right):
# points at each bin's median d(GT, stale), 95% CI bands over clips; changes below
# 0.02 omitted (below the sampler's own resolution).
E.plot_stale_margin_by_size(recs, REF, reveal_window=REVEAL_WINDOW, swing_margin=SWING_MARGIN)


In [ ]:
# %%
# Both state-error readouts stratified by the number of INTERACTIONS fired in the
# clip's exploration leg (pickup/put spans from actions.npz; a move = 2), bins
# 2 / 4 / 6 / 8+. Left: the TAU proxy; right: the stale-margin test. by='k'
# stratifies by station VISITS (meta k_realized, the _k<n>_ id tag) instead.
_ = E.by_interactions(recs, EVAL_DIR, ref=REF, tau=TAU, reveal_window=REVEAL_WINDOW, swing_margin=SWING_MARGIN, min_ref=MIN_REF, by='interactions', bins=(2, 4, 6, 8))


In [ ]:
# %%
# In-mask error and copy rate by the nearest reveal's station type.
E.station_table(recs)

In [ ]:
# %%
# Paired per-clip deltas vs the temporal baseline (clip difficulty removed).
E.paired_deltas(recs, ref="temporal", metric="lpips")

In [ ]:
# %%
# Exemplar mining: worst clips for the FAR arm (`arm` = its record sub-dir) —
# absolute, then relative to temporal (where retrieval LOSES). Paste a CLIP_IDX
# into rollout_ai2thor.
_ = E.worst_clips(recs, arm="pnp_vmz", metric="in_mse", k=10)
print()
_ = E.worst_clips(recs, arm="pnp_vmz", metric="lpips", k=10, ref="temporal")